# Aula 2, RNN

Notebook executável que acompanha a aula [02-rnn.md](../../lessons/modulo-05-deep-learning-nlp/02-rnn.md).

## O que vamos fazer aqui

Vamos construir uma RNN do zero e treiná-la em uma tarefa que isola a memória, lembrar
o primeiro bit de uma sequência binária. O experimento revela a grande limitação da
RNN simples: ela acerta em sequências curtas, mas falha nas longas, porque o gradiente
some ao voltar por muitos passos.

Só numpy. O treino da sequência longa leva alguns segundos.

## RNN com backpropagation no tempo

A função abaixo treina uma RNN para prever o primeiro bit de uma sequência de tamanho
`T`. O passo para frente guarda todos os estados, e o passo para trás desenrola a
sequência, propagando o erro por todos os passos.

In [ ]:
import numpy as np


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def treinar_rnn(T, n_amostras=600, H=12, epocas=120, taxa=0.1, seed=1):
    """Treina uma RNN para lembrar o primeiro bit de uma sequência de tamanho T."""
    rng = np.random.default_rng(seed)
    dados = rng.integers(0, 2, size=(n_amostras, T)).astype(float)
    alvo = dados[:, 0].copy()           # o alvo é o primeiro bit

    Wx = rng.normal(0, 0.3, H); Wh = rng.normal(0, 0.3, (H, H))
    bh = np.zeros(H); Wy = rng.normal(0, 0.3, H); by = 0.0

    for _ in range(epocas):
        for i in rng.permutation(n_amostras):
            seq = dados[i]
            hs = [np.zeros(H)]
            for t in range(T):          # passo para frente, guardando os estados
                hs.append(np.tanh(seq[t] * Wx + hs[-1] @ Wh + bh))
            yhat = sigmoide(hs[-1] @ Wy + by)

            dy = yhat - alvo[i]         # backpropagation no tempo
            Wy -= taxa * dy * hs[-1]; by -= taxa * dy
            dh = dy * Wy
            gWx = np.zeros(H); gWh = np.zeros((H, H)); gbh = np.zeros(H)
            for t in reversed(range(T)):
                draw = dh * (1 - hs[t + 1] ** 2)
                gWx += seq[t] * draw
                gWh += np.outer(hs[t], draw)
                gbh += draw
                dh = draw @ Wh.T
            Wx -= taxa * gWx; Wh -= taxa * gWh; bh -= taxa * gbh

    teste = rng.integers(0, 2, size=(300, T)).astype(float)
    acertos = 0
    for i in range(300):
        h = np.zeros(H)
        for t in range(T):
            h = np.tanh(teste[i, t] * Wx + h @ Wh + bh)
        pred = 1 if sigmoide(h @ Wy + by) > 0.5 else 0
        acertos += int(pred == int(teste[i, 0]))
    return acertos / 300

## Sequência curta contra sequência longa

Agora treinamos a mesma RNN em dois cenários, uma sequência curta de 3 elementos e uma
longa de 25, e comparamos a acurácia em lembrar o primeiro bit.

In [ ]:
print("Sequência curta (T=3) :", round(treinar_rnn(3), 3))
print("Sequência longa (T=25):", round(treinar_rnn(25), 3))

A sequência curta atinge acurácia próxima de 1,0, enquanto a longa fica em
torno de 0,5, ou seja, o mesmo que chutar. Não é falta de capacidade, é o gradiente que
some ao voltar por 25 passos, impedindo a RNN de aprender a dependência longa. Essa é a
motivação direta para a LSTM da próxima aula. Para o projeto, monte a curva de acurácia
por tamanho de sequência.